# E3 · Normalizer is not a scaler: it normalizes the rows, not the columns

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Find out why a KNN on the wines with `Normalizer` "to bring everything onto the same scale" gets 0.79 instead of 0.96: `Normalizer` divides **every row** by its length, erases the magnitude of the samples and does not touch the relative scales of the columns.
Replace it with `StandardScaler` and understand when `Normalizer` is instead the right choice.

## Dataset

`load_wine`: 178 wines, 13 measurements (`proline` has a scale of hundreds, `hue` of 1), 3 classes. Validation: `StratifiedKFold(5, shuffle=True, random_state=0)`.

## Educational purpose

Tell apart two transformations that the name makes you confuse: **scaler** (per column, learned from the training set) and `Normalizer` (per row, stateless). And recognize that "no error and accuracy 0.79" is not a result to accept without a baseline.

## Task

1. Run the colleague's cell: `bug_acc` (KNN with `Normalizer`). Also compute `no_transform_acc` (KNN without transformations) and `std_acc` (with `StandardScaler`): did the colleague improve anything compared to doing nothing?
2. Check what `Normalizer` does: `norms_after` (Euclidean norm of every row after `Normalizer().fit_transform(X)`) and `unit_norms` (True if they are all 1). Save `deviations_after` (`pd.Series`: standard deviation of each column after `Normalizer`), `dominant_column_after` (name of the column with the largest standard deviation) and `scale_ratio` (maximum / minimum standard deviation): were the columns brought to the same scale?
3. Show that the length disappears: `proportional_distance` = Euclidean distance between `Normalizer` applied to `X.iloc[0]` and to `2 * X.iloc[0]` (two wines identical except for a factor 2).
4. Answer with a letter: `what_normalizer_does = 'c'` (a: standardizes the columns; b: brings every column into [0, 1]; c: divides every row by its norm).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, StandardScaler

X, y = load_wine(return_X_y=True, as_frame=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)

## Your work

### The colleague's code

In [ ]:
bug = make_pipeline(Normalizer(), KNeighborsClassifier(7))       # "that way all the columns are on the same scale"
bug_acc = cross_val_score(bug, X, y, cv=cv).mean()
print(f"accuracy with Normalizer: {bug_acc:.3f}")

### Your diagnosis and the fix

In [ ]:
no_transform_acc = None
std_acc = None

In [ ]:
norms_after = None
unit_norms = None
deviations_after = None
dominant_column_after = None
scale_ratio = None
proportional_distance = None
what_normalizer_does = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`Xn = Normalizer().fit_transform(X)`; `np.linalg.norm(Xn, axis=1)`; `pd.DataFrame(Xn, columns=X.columns).std()`. Distance: `Normalizer().transform([X.iloc[0], 2 * X.iloc[0]])` then `np.linalg.norm(a - b)`.'):
    check(bug_acc < 0.85, "the colleague gets less than 0.85", "don't modify the colleague's code")
    check(std_acc is not None and std_acc > 0.94 and std_acc > bug_acc + 0.1, "with StandardScaler the accuracy goes up by over 0.1 (above 0.94)", "make_pipeline(StandardScaler(), KNeighborsClassifier(7))")
    check(no_transform_acc is not None and bug_acc > no_transform_acc, "Normalizer improves on doing nothing, but only a little compared to the scaler: it is an accidental improvement", "no_transform_acc: just KNeighborsClassifier(7)")
    check(unit_norms is True and close(norms_after, 1.0), "after Normalizer every row has norm 1", "np.linalg.norm(..., axis=1)")
    check(dominant_column_after == deviations_after.idxmax() == "magnesium" and scale_ratio > 50, "after Normalizer another column dominates (`magnesium`) and the scales still differ by over 50 times", "std per column of the result")
    check(proportional_distance is not None and proportional_distance < 1e-12, "two wines that differ by a factor 2 have distance 0 after Normalizer", "Normalizer erases the length of the row")
    check(what_normalizer_does == "c", "correct answer: it divides every row by its norm", "c")
done()

---
The commented solution is in `solutions/E_find_the_bug/E3_normalizer_instead_of_standardscaler.ipynb`. Open it after you pass the check, or after you have really tried.